# BUG-24 — Chargeur des données et tests de validation

Ce notebook crée un module Python réutilisable (`src/bug_predictor_data.py`) et une suite de tests automatisés pour les exports produits par `03_preparer_train_validation.ipynb`.

**Règles :** les entrées du modèle sont uniquement `source` et `file_defective` est la cible. Les métadonnées restent disponibles pour l’audit, mais ne sont jamais retournées comme features. Le chargeur de Sprint 2 autorise `train` et `val`; il refuse explicitement `test`. Aucune donnée de test n’est lue ici.

Exécutez d’abord le notebook 03 et vérifiez que Drive contient `processed/time/train_prepared.parquet` et `val_prepared.parquet`. Les petits exemples synthétiques éventuels de la suite sont des fixtures de tests de schéma uniquement; ils ne sont pas utilisés pour analyser ou entraîner le modèle.

In [ ]:
%pip -q install pyarrow

## 1. Monter Drive et vérifier les exports préparés

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path('/content/drive/MyDrive/BugPredictor/defectors_v2')
OUT_DIR = ROOT / 'processed' / 'time'
SRC_DIR = ROOT / 'src'
TEST_DIR = ROOT / 'tests'
SRC_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

for split in ('train', 'val'):
    prepared = OUT_DIR / f'{split}_prepared.parquet'
    if not prepared.is_file():
        raise FileNotFoundError(f'{prepared} absent. Exécutez d’abord 03_preparer_train_validation.ipynb.')
    print(split, prepared, f'{prepared.stat().st_size:,} octets')
print('Le split test ne sera pas ouvert par ce notebook.')

## 2. Générer le module du chargeur

Le module lit les partitions préparées par lots pour limiter la mémoire et valide les colonnes, les valeurs nulles et le caractère binaire de la cible.

In [ ]:
module_code = r'''from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

DEFAULT_ROOT = Path("/content/drive/MyDrive/BugPredictor/defectors_v2")
ALLOWED_SPLITS = frozenset(("train", "val"))
REQUIRED_COLUMNS = frozenset(("source", "file_defective"))


def _partition_path(split, root=None):
    if split not in ALLOWED_SPLITS:
        raise ValueError(
            f"split doit être train ou val; {split!r} est refusé. "
            "Ce chargeur BUG-24 n'ouvre pas test, réservé à l'audit/évaluation indépendante."
        )
    dataset_root = Path(root) if root is not None else DEFAULT_ROOT
    return dataset_root / "processed" / "time" / f"{split}_prepared.parquet"


def _validate_table(table, split="dataset"):
    missing = REQUIRED_COLUMNS - set(table.schema.names)
    if missing:
        raise ValueError(f"Colonnes requises absentes dans {split}: {sorted(missing)}")

    source_type = table.schema.field("source").type
    if not (pa.types.is_string(source_type) or pa.types.is_large_string(source_type)):
        raise TypeError(f"source doit être de type chaîne; type reçu: {source_type}")

    target_type = table.schema.field("file_defective").type
    if not pa.types.is_integer(target_type):
        raise TypeError(f"file_defective doit être entier; type reçu: {target_type}")

    source_column = table.column("source")
    target_column = table.column("file_defective")
    if source_column.null_count:
        raise ValueError(f"{split}: {source_column.null_count} source(s) null(s)")
    if target_column.null_count:
        raise ValueError(f"{split}: {target_column.null_count} label(s) null(s)")

    labels = target_column.to_pylist()
    invalid = sorted({value for value in labels if value not in (0, 1)})
    if invalid:
        raise ValueError(f"{split}: file_defective doit être binaire {{0,1}}; valeurs invalides: {invalid[:10]}")

    if "row_id" in table.schema.names and table.column("row_id").null_count:
        raise ValueError(f"{split}: row_id contient des valeurs nulles")
    return table


def load_split(split, root=None):
    """Charge tout un split train ou val; n'ouvre volontairement jamais test."""
    path = _partition_path(split, root=root)
    if not path.is_file():
        raise FileNotFoundError(f"Fichier préparé introuvable: {path}. Exécutez d'abord le notebook 03.")
    return _validate_table(pq.read_table(path), split=split)


def iter_split_batches(split, batch_size=512, root=None):
    """Itère train ou val par lots pour limiter la mémoire; test est refusé."""
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size doit être un entier strictement positif")
    path = _partition_path(split, root=root)
    if not path.is_file():
        raise FileNotFoundError(f"Fichier préparé introuvable: {path}. Exécutez d'abord le notebook 03.")
    parquet_file = pq.ParquetFile(path)
    missing = REQUIRED_COLUMNS - set(parquet_file.schema_arrow.names)
    if missing:
        raise ValueError(f"Colonnes requises absentes dans {split}: {sorted(missing)}")
    for batch in parquet_file.iter_batches(batch_size=batch_size):
        yield _validate_table(batch, split=split)


def to_xy(table):
    """Retourne uniquement X=source et y=file_defective; les métadonnées sont ignorées."""
    _validate_table(table)
    X = table.column("source").to_pylist()
    y = [int(value) for value in table.column("file_defective").to_pylist()]
    return X, y


def iter_xy(split, batch_size=512, root=None):
    """Itère des couples (X, y) pour train/val, sans charger le split entier."""
    for batch in iter_split_batches(split, batch_size=batch_size, root=root):
        yield to_xy(batch)


def load_train_validation(root=None):
    """Charge les seuls splits autorisés pour le travail train/validation de Sprint 2."""
    return load_split("train", root=root), load_split("val", root=root)
'''
module_path = SRC_DIR / 'bug_predictor_data.py'
module_path.write_text(module_code.strip() + '\n', encoding='utf-8')
print('Module créé :', module_path)
print(module_path.read_text(encoding='utf-8')[:1800])

## 3. Générer et exécuter les tests automatisés

Les tests couvrent le chargement, le streaming, le refus de `test`, l’absence de fichier, les colonnes manquantes, les valeurs nulles, la cible binaire et l’exclusion des métadonnées de X.

In [ ]:
test_code = r'''from pathlib import Path
import sys
import tempfile
import unittest

import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from bug_predictor_data import (  # noqa: E402
    iter_split_batches,
    iter_xy,
    load_split,
    to_xy,
)


class BugPredictorLoaderTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.root = Path(self.temp.name)
        (self.root / "processed" / "time").mkdir(parents=True)

    def tearDown(self):
        self.temp.cleanup()

    def write_split(self, split, data):
        path = self.root / "processed" / "time" / f"{split}_prepared.parquet"
        path.parent.mkdir(parents=True, exist_ok=True)
        pq.write_table(pa.table(data), path)
        return path

    def test_load_and_xy_ignore_metadata_as_features(self):
        self.write_split("train", {
            "row_id": ["train:1", "train:2"],
            "source": ["print('ok')", "raise ValueError()"],
            "file_defective": [0, 1],
            "repo": ["org/demo", "org/demo"],
            "filepath": ["a.py", "b.py"],
        })
        table = load_split("train", root=self.root)
        X, y = to_xy(table)
        self.assertEqual(X, ["print('ok')", "raise ValueError()"])
        self.assertEqual(y, [0, 1])
        self.assertIn("repo", table.column_names)
        self.assertNotIn("repo", X)

    def test_streaming_batches_and_xy(self):
        self.write_split("val", {
            "source": ["a", "b", "c", "d", "e"],
            "file_defective": [0, 1, 0, 1, 0],
        })
        batches = list(iter_split_batches("val", batch_size=2, root=self.root))
        self.assertEqual([batch.num_rows for batch in batches], [2, 2, 1])
        total = sum(len(X) for X, _ in iter_xy("val", batch_size=2, root=self.root))
        self.assertEqual(total, 5)

    def test_test_split_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "train ou val"):
            load_split("test", root=self.root)

    def test_missing_file_is_reported(self):
        with self.assertRaises(FileNotFoundError):
            load_split("train", root=self.root)

    def test_missing_required_column_is_rejected(self):
        self.write_split("train", {"source": ["x"]})
        with self.assertRaisesRegex(ValueError, "file_defective"):
            load_split("train", root=self.root)

    def test_null_source_is_rejected(self):
        self.write_split("train", {
            "source": ["ok", None],
            "file_defective": [0, 1],
        })
        with self.assertRaisesRegex(ValueError, "source.*null"):
            load_split("train", root=self.root)

    def test_null_target_is_rejected(self):
        self.write_split("train", {
            "source": ["ok", "bad"],
            "file_defective": pa.array([0, None], type=pa.int8()),
        })
        with self.assertRaisesRegex(ValueError, "label.*null"):
            load_split("train", root=self.root)

    def test_non_binary_target_is_rejected(self):
        self.write_split("train", {
            "source": ["x"],
            "file_defective": [2],
        })
        with self.assertRaisesRegex(ValueError, "binaire"):
            load_split("train", root=self.root)

    def test_invalid_batch_size_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "batch_size"):
            list(iter_split_batches("train", batch_size=0, root=self.root))


if __name__ == "__main__":
    unittest.main()
'''
test_path = TEST_DIR / 'test_bug_predictor_data.py'
test_path.write_text(test_code.strip() + '\n', encoding='utf-8')
result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', str(TEST_DIR), '-p', 'test_bug_predictor_data.py', '-v'],
    cwd=str(ROOT), text=True, capture_output=True
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Les tests BUG-24 ont échoué (exit {result.returncode}).')
print('Tous les tests BUG-24 ont réussi.')

## 4. Valider les exports réels par lots

Cette étape parcourt uniquement train et validation, compte les classes et écrit un rapport de validation du chargeur. Elle n’entraîne aucun modèle.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from bug_predictor_data import iter_xy

summary = {}
for split in ('train', 'val'):
    rows = positives = negatives = 0
    for X, y in iter_xy(split, batch_size=512, root=ROOT):
        rows += len(X)
        positives += sum(value == 1 for value in y)
        negatives += sum(value == 0 for value in y)
    summary[split] = {
        'rows_validated': rows,
        'positive': positives,
        'negative': negatives,
        'features': ['source'],
        'target': 'file_defective'
    }
    print(split, summary[split])

report = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'module': str(module_path),
    'tests': str(test_path),
    'partitions_validated': summary,
    'test_partition': 'not opened; excluded from this train/validation loader',
    'model_training_performed': False
}
report_path = ROOT / 'audit' / 'bug24_loader_validation.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Rapport de validation :', report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Livrables produits dans Google Drive

- `src/bug_predictor_data.py` — chargeur réutilisable train/validation.
- `tests/test_bug_predictor_data.py` — tests `unittest`.
- `audit/bug24_loader_validation.json` — preuve des tests et des contrôles sur les exports préparés.

Conservez le test hors du chargement et de tout réglage du modèle. L’évaluation indépendante sera planifiée dans Sprint 3, avec un protocole distinct.